# LLM 症状映射实验

# 1.实验背景

   慢性淋巴细胞白血病患者的症状表现复杂多样，需要构建一个症状到证型、治法的自动映射流程。本实验要求完成从数据读取到结果生成的全流程实现。

# 2.实验目标

   理解症状映射任务的基本原理与方法

   掌握数据读取、处理与写入的全流程操作

   探索不同的提示工程与映射策略

   实现可扩展、可解释的文本映射流程

# 3.数据与任务

1. 数据准备

    使用提供的result.csv文件作为数据源，包含患者症状描述
    
    文件路径：result.csv
    
    数据格式：CSV格式，包含序号、症状两列
    
2. 解决方案设计

    开放设计空间：可采用规则引擎、机器学习模型或其他创新方法

# 4.基线实现

In [1]:
import json
import requests
import sys


## 4.1 调用大模型生成标签结果

In [34]:
import csv
import json
import os
import re
from openai import OpenAI
import time

# 配置大模型API参数，请在本地环境变量中设置 BAIDU_LLM_API_KEY
client = OpenAI(
    api_key=os.getenv("BAIDU_LLM_API_KEY", ""),
    base_url=os.getenv("BAIDU_LLM_BASE_URL", "https://aistudio.baidu.com/llm/lmapi/v3")
)


In [35]:
def parse_response(content):
    """
    解析大模型返回的响应内容
    处理包含代码块的JSON响应
    """
    # 检查是否包含代码块
    # print(content)
    code_block_pattern = re.compile(r"```json\s*(.*?)\s*```", re.DOTALL)
    # print(code_block_pattern)
    match = code_block_pattern.search(content)

    if match:
        json_str = match.group(1).strip()
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            print(f"JSON解析失败: {json_str}")
            return None
    else:
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            print(f"直接JSON解析失败: {content}")
            return None


In [41]:
def prompt_engineering_infer(symptoms):
    """
    调用大模型进行结构化标签生成
    返回结构化JSON结果
    """
    system_prompt = """
    你是一位经验丰富的领域专家，请根据患者症状描述判断：
    1. 证型：需符合任务定义中的标准标签
    2. 治法：需与证型对应且符合领域规则
    请用JSON格式输出：{"证型":"", "治法":""}
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"患者症状：{symptoms}"}
    ]

    try:

        response = client.chat.completions.create(
            model="ernie-4.5-0.3b",
            messages=messages,
            max_completion_tokens=512,
            temperature=0.3,
            top_p=0.7
        )

        # 解析响应内容
        content = response.choices[0].message.content
        result = parse_response(content)

        if result:
            # print(result)
            return result
        else:
            return {"证型": "未知", "治法": "待定"}

    except Exception as e:
        print(f"API调用异常: {str(e)}")
        return {"证型": "未知", "治法": "待定"}


In [48]:
def prompt_engineering_infer(symptoms):
    """
    调用大模型进行结构化标签生成
    返回结构化JSON结果
    """
    system_prompt = """
你是一位经验丰富并谨慎的领域专家。任务：根据患者症状判断【证型】并给出对应【治法】。请严格遵守下面格式与要求：

输出格式（必须为严格的JSON对象，允许包含在```json```代码块中）：
{
  "证型": "短文本（必填）",从这里选：痰湿内蕴、脾虚痰湿、气阴两虚、痰瘀互结、痰湿内蕴兼气虚发热,
  "治法": "短文本（必填）",
  "理由": "不超过40字的简要诊断依据（必填）",
  "置信度": 0.00,   // 一个0-1之间的小数，表示你对该诊断的置信度（必填）
  "自评分": 0        // 按下列评分规则对你本次输出打分（0-100整数，必填）
}

评分规则（用于生成自评分）：
- 诊断与任务标签空间符合：40分
- 治法与证型对应、符合领域原则：30分
- 理由简洁且包含关键证据（如：脾虚、痰湿、气虚、血瘀等）：20分
- 置信度与自评分一致性（置信度*100 与 自评分差距≤20）：10分

注意：
1）不要输出内部链式思考（不要输出长篇推理过程）。只给出上面指定的结构化字段和简短理由。  
2）理由应为“诊断依据”的摘要，不超过40字（例如：“脾虚兼湿，舌淡苔白，倦怠腹胀”）。  
3）置信度与自评分是量化提示，目的是促使模型自检；请诚实评估。  
4）如果无法判断，请返回证型为\"未知\"并把置信度设为0，自评分设为0，理由写\"信息不足\"。
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"患者症状：{symptoms}"}
    ]

    try:

        response = client.chat.completions.create(
            model="ernie-4.5-0.3b",
            messages=messages,
            max_completion_tokens=512,
            temperature=0.3,
            top_p=0.7
        )

        # 解析响应内容
        content = response.choices[0].message.content
        result = parse_response(content)

        if result:
            # print(result)
            return result
        else:
            return {"证型": "未知", "治法": "待定"}

    except Exception as e:
        print(f"API调用异常: {str(e)}")
        return {"证型": "未知", "治法": "待定"}

In [44]:
# 提示工程增强版
def prompt_engineering_infer(symptoms):
    
    system_prompt = """
    你是一位经验丰富的领域专家，专门研究该任务中的症状映射与标签生成。

    【任务要求】
    请根据患者症状描述准确判断：
    1. 证型：必须符合任务定义中的标准标签
    2. 治法：必须与证型严格对应且符合领域原则

    【证型选择范围】
    请从以下标准证型中选择最匹配的：
    - 痰湿内蕴（表现为咳嗽、痰多、痰白黏腻、胸闷、脘腹胀满、食欲不振、舌苔白腻、脉滑。）
    - 脾虚痰湿 （表现为脾虚症状如乏力、食欲不振、腹泻、腹胀，加上痰湿症状如痰多、咳嗽、舌苔白腻、脉滑。）
    - 气阴两虚（表现为气虚症状如乏力、气短、自汗，阴虚症状如口干、咽干、潮热、盗汗、舌红少苔、脉细数）
    - 痰瘀互结（表现为痰湿症状如痰多、胸闷，加上瘀血症状如疼痛、肿块、舌质紫暗、有瘀点、脉涩。）
    - 痰湿内蕴兼气虚发热（表现为湿内蕴症状如痰多、胸闷、舌苔腻，加上气虚发热症状如低热、乏力、自汗、气短、舌淡苔白、脉虚数。）

    【输出格式】
    严格使用JSON格式：{"证型":"", "治法":""}

    【思考过程】
    请先分析症状特点，再对照证型标准，最后确定治法和方药原则。
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"患者症状：{symptoms}"}
    ]

    try:

        response = client.chat.completions.create(
            model="ernie-4.5-0.3b",
            messages=messages,
            max_completion_tokens=512,
            temperature=0.3,
            top_p=0.7
        )

        # 解析响应内容
        content = response.choices[0].message.content
        result = parse_response(content)

        if result:
            # print(result)
            return result
        else:
            return {"证型": "未知", "治法": "待定"}

    except Exception as e:
        print(f"API调用异常: {str(e)}")
        return {"证型": "未知", "治法": "待定"}

In [45]:
# 读取训练数据中的症状、证型和治法
train_data = './train_data.csv'
symptoms_data = []
ZX = []  # 证型
ZF = []  # 治法

with open(train_data, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        symptoms_data.append(row[1])  # 提取症状列
        ZX.append(row[2])  # 提取症状列
        ZF.append(row[3])  # 提取症状列



In [7]:
symptoms_data[0]

'患者男性，身高170cm，体重70kg。 \n西医诊断为慢性淋巴细胞白血病。 \n现症见：口中发黏，唾液粘稠、痰涎量多，偶有咯吐、偶有发生、时有胸闷或腹胀，可自行缓解、劳时即乏。 舌象可见淡红、齿痕、淡黄、润，脉象为弦脉。 '

In [38]:
ZX

['痰湿内蕴',
 '痰湿内蕴',
 '痰湿内蕴',
 '脾虚痰湿',
 '气阴两虚',
 '脾虚痰湿',
 '气阴两虚',
 '气阴两虚',
 '脾虚痰湿',
 '脾虚痰湿',
 '气阴两虚',
 '痰湿内蕴',
 '脾虚痰湿',
 '脾虚痰湿',
 '脾虚痰湿',
 '痰瘀互结',
 '痰湿内蕴兼气虚发热']

In [10]:
ZF[1]

'健脾燥湿，化痰利浊'

根据症状生成相应的证型及治法

In [49]:
# 生成新的结果数据
predict_zx = []
predict_zf = []

def batch_predict(symptom):
    model_result = prompt_engineering_infer(symptom)
    return model_result['证型'], model_result['治法']


for symptom in symptoms_data:
    zx, zf = batch_predict(symptom)
    predict_zx.append(zx), predict_zf.append(zf)
    time.sleep(0.2)  # 调用间隔，避免过快请求导致API限制


In [50]:
predict_zx

['痰湿内蕴',
 '痰湿内蕴',
 '痰湿内蕴',
 '痰湿内蕴',
 '痰湿内蕴',
 '脾虚痰湿',
 '痰湿内蕴',
 '脾虚痰湿',
 '脾虚痰湿',
 '痰湿内蕴',
 '脾虚痰湿',
 '脾虚痰湿',
 '脾虚痰湿',
 '痰湿内蕴',
 '痰湿内蕴',
 '痰湿内蕴',
 '脾虚痰湿']

In [15]:
predict_zf[1]

'健脾益气，化痰散结'

查看输出的证型，治法

In [ ]:
predict_zx


In [ ]:
predict_zf


## 4.2 评估生成结果

调用 embedding 模型接口，生成文本嵌入表示

In [16]:
def encode_text(text):
    """
    获取文本的向量表示
    """
    url = "https://qianfan.baidubce.com/v2/embeddings"
    payload = json.dumps({
        "model": "bge-large-zh",
        "input": [text]
    })
    headers = {'Content-Type': 'application/json'}
    embedding_api_key = os.getenv('BAIDU_EMBEDDING_API_KEY', '')
    if embedding_api_key:
        headers['Authorization'] = f'Bearer {embedding_api_key}'

    try:
        response = requests.post(url, headers=headers, data=payload, timeout=10)
        response.raise_for_status()
        data = response.json()
        return data['data'][0]['embedding']
    except Exception as e:
        print(json.dumps({
            "code": 1,
            "errorMsg": f"Embedding API error: {str(e)}",
            "score": 0.0,
            "data": [{"score": 0}]
        }, ensure_ascii=False), flush=True)


测试嵌入编码效果

In [ ]:
encode_text('how old are you')


计算文本相似度

In [18]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def semantic_similarity(text1, text2):
    """
    计算两个文本的相似度
    """
    if not text1.strip() or not text2.strip():
        return 0.0
    try:
        emb1 = encode_text(text1)
        emb2 = encode_text(text2)
        sim = cosine_similarity(
            np.array(emb1).reshape(1, -1),
            np.array(emb2).reshape(1, -1)
        )
        return float(sim[0][0])
    except Exception as e:
        print(json.dumps({
            "code": 1,
            "errorMsg": f"Similarity calculation error: {str(e)}",
            "score": 0.0,
            "data": [{"score": 0}]
        }, ensure_ascii=False), flush=True)


计算证型、治法真实值与预测值的余弦相似度，取平均作为最终分数。

In [19]:
def evaluate_predictions(predict_zx, predict_zf):
    ZX_score = []
    ZF_score = []
    for i in range(len(predict_zx)):

        zheng_xing_sub = predict_zx[i].strip()
        zheng_xing_gro = ZX[i].strip()
        zhi_fa_sub = predict_zf[i].strip()
        zhi_fa_gro = ZF[i].strip()

        sim_x = semantic_similarity(zheng_xing_sub, zheng_xing_gro)
        sim_f = semantic_similarity(zhi_fa_sub, zhi_fa_gro)

        ZX_score.append(sim_x)
        ZF_score.append(sim_f)

    zx_mean = np.mean(ZX_score) if ZX_score else 0.0
    zf_mean = np.mean(ZF_score) if ZF_score else 0.0

    final_score = ((zx_mean + zf_mean) / 2) * 100  # 百分制
    return final_score


In [20]:
final_score = evaluate_predictions(predict_zx, predict_zf)
print(final_score)


87.35767420842703


# 5. 结果评估

**说明**：    
                         
1. 根据提供的大模型 API，实现症状到标签的结构化映射。


**注意事项：**
1. 点击左侧栏`提交结果`后点击`生成文件`则只需勾选 `predict()`及必要的相关函数的cell。
2. 请导入必要的包和第三方库 (包括此文件中曾经导入过的)。
3. `predict()`函数的输入和输出请**不要改动**。

============================  **模型推理代码答题区域**  ============================
<br>
在下方的代码块中编写 **模型预测** 部分的代码，请勿在别的位置作答

In [ ]:
from openai import OpenAI
#  实现大模型调用方法

#  进行推理输出相应内容
def predict(symptom):
    """
    :param symptom: str, 症状描述
    :return zx_predict, zf_predict：str, 依次为症型，治法描述
    """
    #  根据输入的症状，使用大模型推理相应的内容


    return zx_predict, zf_predict
